# 第4回：データ探偵—分布・欠損・外れ値

**今日の問い：モデルを作る前に、データの怪しいところをどう見つけるか。**

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

`TRY`は全員、`CHANGE`は値を1つ変える練習、`CHALLENGE`は余裕がある人向けです。
`DEEP DIVE`・`APPENDIX`は経験者や自習向けの発展で、飛ばしても本編は完結します。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 単変量・二変量・群別の順でデータを見る
- 欠損の発生機構と外れ値を、検定や多変量手法で客観的に調べる
- 図と統計量から、断定ではなく検証可能な仮説を作る

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 分布：値がどこにどれだけ存在するか
- 外れ値：他と大きく異なる観測値
- 相互情報量：非線形も捉える関連の強さ
- 欠損機構：MCAR/MAR/MNARという欠損の起こり方
- 多変量外れ値：単変量では見えない組み合わせの異常

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## EDA＝モデルを作る前にデータをよく見る工程

EDA（探索的データ分析）は、いきなりモデルを作らず、まずデータをよく見る工程です。目的は
「きれいなグラフを作ること」ではなく、**モデルを惑わせる怪しい点（偏り・欠損・外れ値）を先に見つけ、
検証できる仮説を作ること**です。

見る順番にはコツがあります：**1変数（分布）→ 2変数（関係）→ 群別（カテゴリごと）**。
いきなり複雑な図に行かず、単純な図から積み上げます。次のセルはまず描画の下準備（日本語表示と
見た目のテーマ設定）です。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
# グラフの日本語が文字化けしないようにする設定です。中身は今は理解しなくてOK、そのまま実行してください。
import matplotlib.pyplot as plt
from matplotlib import font_manager
for _name in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]:
    if _name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _name
        break
import seaborn as sns
sns.set_theme(style="whitegrid")


## TRY：1変数の分布を見る（ヒストグラムと箱ひげ図）

まず1列ずつ「値がどこに、どれだけあるか」を見ます。

- **ヒストグラム**：値を区間に分け、各区間の件数を棒で表す。山の形・偏り・飛び離れた値が見えます。
- **箱ひげ図**：中央値・四分位・外れ値候補（ひげの外の点）をコンパクトに表す。外れ値探しに向きます。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="yield_pct", bins=20, ax=axes[0])
axes[0].set_title("収率の分布")
sns.boxplot(data=df, x="reaction_time_h", ax=axes[1])
axes[1].set_title("反応時間：外れ値候補を探す")
plt.tight_layout()


### 出力の読み方

- **左（収率のヒストグラム）**：山が1つか2つか、左右どちらに裾を引くかを見ます。裾が長い＝一部に極端な値。
- **右（反応時間の箱ひげ図）**：箱が中央50%、ひげの外の点が外れ値候補。**右端にぽつんと離れた点**があれば、それが要調査の試料です（このデータには意図的に極端な値を仕込んであります）。
- まだ「削除」はしません。EDAは**見つける**段階です。


## 2変数の関係とカテゴリ比較（散布図・箱ひげ図）

次に「2つの列の関係」を見ます。散布図は連続値どうしの関係、色分け（`hue`）で3つ目の情報（触媒）も
重ねられます。カテゴリごとの違いは箱ひげ図で比べます。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="temperature_c", y="yield_pct", hue="catalyst", alpha=0.65, ax=axes[0])
axes[0].set_title("温度と収率")
sns.boxplot(data=df, x="catalyst", y="yield_pct", ax=axes[1])
axes[1].set_title("触媒別の収率")
plt.tight_layout()


### 出力の読み方

- **左（温度×収率）**：右肩上がりの直線ではなく、**中くらいの温度で収率が高くなる山型**に見えるはずです。「関係＝直線」とは限らないことを、目で確認しておきます（第4回DEEP DIVEの相互情報量につながります）。
- 点の色（触媒）で**かたまり**ができていれば、触媒が収率に効いている手がかり。
- **右（触媒別の箱ひげ）**：触媒ごとに箱の高さ（収率の中心）が違えば、触媒の効果が疑われます。ただし件数が少ない触媒は割り引いて読みます。


## TRY：欠損と「明らかに怪しい値」を表で押さえる

図で当たりを付けたら、表で具体的に特定します。どの列にいくつ欠損があるか、そして温度が異常に
大きい上位5件を実際に取り出します。


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("欠損数"))
display(df.nlargest(5, "temperature_c")[["sample_id", "temperature_c", "reaction_time_h", "yield_pct"]])


### 出力の読み方

- **1つ目の表**：欠損のある列と件数。欠損の多い列は、後で「埋める／落とす／別扱い」の判断が要ります。
- **2つ目の表**：温度が高い順の5件。`180.0`のような**周囲から突出した値**があれば、入力ミスか特殊な実験かを疑い、`sample_id`を控えて確認先を考えます。
- ここでも即削除しないのが鉄則。**「誰に確認するか」「残した場合に何が起きるか」**まで考えてから対処します。


## CHANGE

散布図の色分け（`hue`）を`catalyst`から`solvent`へ変え、見え方の違いを1つ挙げます。

## 注意

外れ値＝入力ミス、ではありません。本物の珍しい現象のこともあります。EDAの結論は「削除」ではなく、
**「確認すべき仮説」**の形で残します。


## DEEP DIVE：印象を統計量で裏づける

図の印象は主観的です。ここでは4つの道具で客観化します：**相関**（直線的な関係）、
**相互情報量**（曲がった関係も拾う）、**欠損機構**（欠損の起こり方）、**多変量外れ値**（組み合わせの異常）。


### 相関ヒートマップ：全列の関係を一望する

`corr()`は数値列すべての**相関係数**（-1〜+1）を計算します。+1に近いほど一緒に増え、-1に近いほど
片方が増えると片方が減る関係。色の濃淡で一望できます。ただし**相関は「直線的な」関係しか測れない**
ことに注意します。


In [ ]:
numeric_cols = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa", "yield_pct"]
correlation = df[numeric_cols].corr()
plt.figure(figsize=(8, 5))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("数値列の相関（因果ではない）")
plt.tight_layout()


### 出力の読み方

- 対角線は自分自身との相関で必ず1.00。赤いマスほど正、青いマスほど負の相関。
- `temperature_c`と`yield_pct`の相関は**意外と弱い**はずです（山型の関係なので直線相関では捉えきれない）。
- **重要**：相関は因果ではありません。「AとBが一緒に動く」ことと「AがBの原因」は別物です。


### 相互情報量：曲がった関係も拾う

温度のように「最適点で収率が最大」の山型は、相関では弱く見えます。**相互情報量**は直線に限らず
「片方を知るともう片方の予想がどれだけ絞れるか」を測るので、こうした関係を拾えます。相関と並べて読みます。


In [ ]:
from sklearn.feature_selection import mutual_info_regression

mi_source = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
mi_frame = df[mi_source + ["yield_pct"]].dropna()
mi = mutual_info_regression(mi_frame[mi_source], mi_frame["yield_pct"], random_state=42)
pearson = mi_frame[mi_source].corrwith(mi_frame["yield_pct"]).abs()
compare = pd.DataFrame({"相互情報量": mi, "|相関|": pearson.to_numpy()}, index=mi_source)
compare.sort_values("相互情報量", ascending=False).round(3)


### 出力の読み方

`temperature_c`は**相互情報量は大きいのに|相関|は小さい**、という食い違いが見えるはずです。これが
「相関だけで特徴量を捨ててはいけない」理由です。両方を見て、関係の形は散布図で確かめます。


### 欠損の起こり方（欠損機構）を疑う

欠損はランダムとは限りません。**MCAR**（完全にランダム）、**MAR**（他の列で説明できる偏り）、
**MNAR**（値そのものに依存）で対処が変わります。ここでは「温度の欠損率が溶媒で偏るか」を見ます。


In [ ]:
miss = df.assign(temp_missing=df["temperature_c"].isna())
by_solvent = miss.groupby("solvent", dropna=False)["temp_missing"].mean().round(3)
print("溶媒別の温度欠損率:")
print(by_solvent)
print("溶媒でほぼ一定ならMCARに近い。偏るならMARを疑う。")


### 出力の読み方

溶媒によって温度欠損率が大きく違えば、欠損は溶媒と関係している（MARの疑い）＝
「一律に中央値で埋める」のが危ういサインです。値がほぼ一定なら、単純な補完でも大きな害は出にくいと判断できます。


### 多変量外れ値：組み合わせの異常を探す

「温度は普通、時間も普通、でもその組み合わせは他にない」という試料は、1列ずつ見ても見つかりません。
`IsolationForest`は**複数列を同時に見て、周囲から孤立した点**を外れ値候補として検出します。


In [ ]:
from sklearn.ensemble import IsolationForest

iso_cols = ["temperature_c", "reaction_time_h", "concentration_m", "yield_pct"]
iso_data = df[iso_cols].fillna(df[iso_cols].median())
flags = IsolationForest(contamination=0.03, random_state=42).fit_predict(iso_data)
outliers = df.loc[flags == -1, ["sample_id", *iso_cols]]
print("多変量外れ値候補:", len(outliers), "件")
outliers.round(2)


### 出力の読み方

- `contamination=0.03`は「全体の約3%を外れ値候補とみなす」設定です（多すぎ・少なすぎると感じたら調整）。
- 出た試料を1件ずつ見て、**どの列の組み合わせが変か**を考えます。単変量の箱ひげ図では正常だった試料が混じっていれば、多変量で見る価値があった、ということです。
- ここでも自動削除はせず、確認対象のリストとして扱います。


## APPENDIX（任意・追加演習）

可視化の引き出しを増やします。90分の外の自習向けです。まず**pairplot**で、複数の数値列の関係を
一度に俯瞰します（散布図と分布のマトリクス）。


In [ ]:
subset = df[["temperature_c", "reaction_time_h", "yield_pct", "catalyst"]].dropna()
sns.pairplot(subset, hue="catalyst", corner=True, plot_kws={"alpha": 0.5})


### 出力の読み方

対角線は各列の分布、対角線以外は2列の散布図で、色は触媒。**触媒ごとにかたまりができている**列の組が
あれば、それが効いている手がかり。多くの列を一気に眺めて当たりを付け、気になったペアを個別の図で
深掘りします（俯瞰→詳細の順）。


### バイオリン図とカウント図

**バイオリン図**は箱ひげ図より分布の形（山が1つか2つか）が分かります。**カウント図**はカテゴリの件数。
分布の形と件数を押さえると、平均の解釈がぐっと安全になります。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.violinplot(data=df, x="catalyst", y="yield_pct", ax=axes[0])
axes[0].set_title("触媒別の収率分布（バイオリン）")
sns.countplot(data=df, x="solvent", ax=axes[1])
axes[1].set_title("溶媒の件数")
plt.tight_layout()


### 出力の読み方

- **バイオリン**：横幅が太い高さに値が集まっています。二山（2つのふくらみ）なら、隠れた別グループの存在を疑います。
- **カウント図**：件数の少ない溶媒は、以降の群別集計で平均が不安定になりやすい箇所。分析前に把握しておきます。


### 群別の目的変数と、目的変数との関連を棒で見る

「触媒別の活性率」と「収率との|相関|が強い列」を棒グラフで並べます。EDAの締めとして、
**目的変数（active/yield）に効きそうな列**の当たりを付けます。


In [ ]:
active_rate = df.groupby("catalyst")["active"].mean().sort_values(ascending=False)
corr_target = df.select_dtypes("number").corr()["yield_pct"].drop("yield_pct").abs().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
active_rate.plot.bar(ax=axes[0], title="触媒別の活性率")
corr_target.plot.bar(ax=axes[1], title="収率との|相関|")
plt.tight_layout()


### 出力の読み方

- **左**：触媒によって活性率が違えば、触媒は分類（active）に効く候補。
- **右**：収率との|相関|が高い列が、回帰（yield）で効く候補。ただし第4回本編のとおり、相関が低くても相互情報量が高い列（温度など）を見落とさないよう、相関の棒だけで判断しないこと。
- ここで挙がった候補列が、第5回以降の特徴量選びの出発点になります。


## よくある誤り

- 外れ値を自動削除する
- 相関を因果と読む
- 見栄えの良い図だけを選ぶ

## SELF-STUDY（任意・30〜60分）

- 相互情報量の上位3列について、散布図で関係の形を確認する
- IsolationForestの外れ値候補2件を、残す場合と除く場合で整理する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 相関係数と相互情報量はどう違うか
2. MCARとMARの違いは何か
3. 多変量外れ値が単変量で見つからない理由は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
